### Завдання 1. Завантаження даних NOAA
Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних;

In [1]:
import urllib.request
import os
import datetime
import pandas as pd
import glob
from IPython.display import display

os.makedirs('vhi_data', exist_ok=True)

def download_vhi_data(province_id):
    now = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"vhi_data/vhi_province_{province_id}_{now}.csv"
    url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
    
    try:
        urllib.request.urlretrieve(url, filename)
    except Exception as e:
        print(e)

existing_files = glob.glob("vhi_data/vhi_province_*.csv")

if len(existing_files) < 27:
    print("Завантажуємо файли з сервера NOAA")
    for i in range(1, 28):
        download_vhi_data(i)
    print("Завантаження завершено.")
else:
    print("Файли вже існують")

Завантажуємо файли з сервера NOAA
Завантаження завершено.


### Завдання 2. Очищення даних (Data Cleaning)
Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області

In [2]:
def read_and_clean_vhi_data(directory_path='vhi_data'):
    files = glob.glob(f"{directory_path}/*.csv")
    dataframes = []
    
    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty']
    
    for file in files:
        province_id = int(file.split('vhi_province_')[1].split('_')[0])
        
        df = pd.read_csv(file, header=1, names=headers, skipinitialspace=True)
        
        df.drop(columns=['empty'], inplace=True, errors='ignore')
        
        df.dropna(subset=['VHI'], inplace=True)
        
        df['Year'] = df['Year'].astype(str).str.replace(r'<tt><pre>', '', regex=True)
        df = df[df['Year'] != '']
        df['Year'] = df['Year'].astype(int)
        
        df['Old_Province_ID'] = province_id
        
        dataframes.append(df)
    
    combined_dataframe = pd.concat(dataframes, ignore_index=True)
    return combined_dataframe

df_vhi = read_and_clean_vhi_data()
print("Загальна кількість записів")
print(len(df_vhi))
display(df_vhi.head())

Загальна кількість записів
60372


,Year,Week,SMN,SMT,VCI,TCI,VHI,Old_Province_ID
0,1982,1.0,0.059,258.24,51.11,48.78,49.95,10
1,1982,2.0,0.063,261.53,55.89,38.20,47.04,10
2,1982,3.0,0.063,263.45,57.30,32.69,44.99,10
3,1982,4.0,0.061,265.10,53.96,28.62,41.29,10
4,1982,5.0,0.058,266.42,46.87,28.57,37.72,10


### Завдання 3. Зміна індексів та додавання назв
Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька). 

In [3]:
def reindex_and_add_names(dataframe):
    mapping_dict = {
        1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21, 
        11: 9, 12: 26, 13: 10, 14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 
        20: 27, 21: 17, 22: 18, 23: 6, 24: 1, 25: 2, 26: 7, 27: 5
    }
    
    names_dict = {
        1: 'Вінницька', 2: 'Волинська', 3: 'Дніпропетровська', 4: 'Донецька', 5: 'Житомирська',
        6: 'Закарпатська', 7: 'Запорізька', 8: 'Івано-Франківська', 9: 'Київська', 10: 'Кіровоградська',
        11: 'Луганська', 12: 'Львівська', 13: 'Миколаївська', 14: 'Одеська', 15: 'Полтавська',
        16: 'Рівненська', 17: 'Сумська', 18: 'Тернопільська', 19: 'Харківська', 20: 'Херсонська',
        21: 'Хмельницька', 22: 'Черкаська', 23: 'Чернівецька', 24: 'Чернігівська', 25: 'Республіка Крим',
        26: 'м. Київ', 27: 'м. Севастополь'
    }
    
    dataframe['Province_ID'] = dataframe['Old_Province_ID'].map(mapping_dict)
    dataframe['Province_Name'] = dataframe['Province_ID'].map(names_dict)
    
    dataframe.drop(columns=['Old_Province_ID'], inplace=True)
    return dataframe

df_vhi_updated = reindex_and_add_names(df_vhi)
print("Індекси змінено")
display(df_vhi_updated[['Year', 'Week', 'VHI', 'Province_ID', 'Province_Name']].head())

Індекси змінено


,Year,Week,VHI,Province_ID,Province_Name
0,1982,1.0,49.95,21,Хмельницька
1,1982,2.0,47.04,21,Хмельницька
2,1982,3.0,44.99,21,Хмельницька
3,1982,4.0,41.29,21,Хмельницька
4,1982,5.0,37.72,21,Хмельницька


### Завдання 4.1. Формування вибірок
Реалізувати процедури для формування вибірок наступного виду:
Ряд VHI для області за вказаний рік;


In [4]:
def get_vhi_series_by_year(dataframe, province_id, year):
    condition1 = dataframe['Province_ID'] == province_id
    condition2 = dataframe['Year'] == year
    
    filtered_data = dataframe[condition1 & condition2]
    return filtered_data[['Week', 'VHI']]

print("Вибірка: VHI для Київської області (ID 9) за 2020 рік:")
result_4_1 = get_vhi_series_by_year(df_vhi_updated, 9, 2020)
display(result_4_1.head())

Вибірка: VHI для Київської області (ID 9) за 2020 рік:


,Week,VHI
4212,1.0,37.78
4213,2.0,38.41
4214,3.0,39.74
4215,4.0,41.90
4216,5.0,43.53


### Завдання 4.2. Формування вибірок
Ряд VHI за вказаний діапазон років для вказаних областей

In [5]:
def get_vhi_series_by_years_and_regions(dataframe, province_ids, start_year, end_year):
    condition1 = dataframe['Province_ID'].isin(province_ids)
    condition2 = dataframe['Year'] >= start_year
    condition3 = dataframe['Year'] <= end_year
    
    filtered_data = dataframe[condition1 & condition2 & condition3]
    return filtered_data[['Year', 'Week', 'Province_Name', 'VHI']]

print("Вибірка: VHI для Львівської (12) та Одеської (14) областей за 2018-2019 роки:")
result_4_2 = get_vhi_series_by_years_and_regions(df_vhi_updated, [12, 14], 2018, 2019)
display(result_4_2.head())

Вибірка: VHI для Львівської (12) та Одеської (14) областей за 2018-2019 роки:


,Year,Week,Province_Name,VHI
13052,2018,1.0,Львівська,47.91
13053,2018,2.0,Львівська,48.77
13054,2018,3.0,Львівська,50.34
13055,2018,4.0,Львівська,50.10
13056,2018,5.0,Львівська,49.55


### Завдання 4.3. Пошук екстремумів
Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани.


In [6]:
def get_vhi_statistics(dataframe, province_ids, years):
    condition1 = dataframe['Province_ID'].isin(province_ids)
    condition2 = dataframe['Year'].isin(years)
    
    filtered_data = dataframe[condition1 & condition2]
    
    vhi_min = filtered_data['VHI'].min()
    vhi_max = filtered_data['VHI'].max()
    vhi_mean = filtered_data['VHI'].mean()
    vhi_median = filtered_data['VHI'].median()
    
    print("Статистика VHI:")
    print("Мінімум:")
    print(vhi_min)
    print("Максимум:")
    print(vhi_max)
    print("Середнє арифметичне:")
    print(vhi_mean)
    print("Медіана:")
    print(vhi_median)

print("Статистика для Харківської (19) та Донецької (4) областей за 2015 та 2021 роки:")
get_vhi_statistics(df_vhi_updated, [4, 19], [2015, 2021])

Статистика для Харківської (19) та Донецької (4) областей за 2015 та 2021 роки:
Статистика VHI:
Мінімум:
26.55
Максимум:
75.78
Середнє арифметичне:
47.026250000000005
Медіана:
45.355
